In [ ]:
import wormtrails as wts
import cv2
import numpy as np
import pandas as pd

# Basic sequential processing pipeline

In [ ]:
# Sequential processing (whole video fits in memory)
video_path = "./lifespan_plate.avi"

# Load the raw video scan
video_array = wts.read_video_file(video_path)

# Preprocessing of the scan
corrected_array = wts.correct_vignetting(video_array)

# For most cases, we recommend using subtract_average
motion_array = wts.subtract_average(corrected_array)

In [ ]:
# Preview your visualization parameters
wts.show_time_encoding(motion_array, window=20, scale_factor=20, offset=-20, light_background=True)

In [ ]:
# Create and save your visualization
vis = wts.create_time_encoded_frame(
    motion_array, 
    colormap=wts.hsv_rainbow, 
    window=30, 
    start_time=0, 
    scale_factor=20, 
    offset=-20, 
    light_background=True, # sets whether the output visualization will have a light or dark background, not whether the input did
    mode='auto' # can be set to 'vectorized' if there is surplus memory available
)
wts.show_frame(vis)
cv2.imwrite("./rainbow.png", vis)

In [ ]:
# This example creates a video instead of a single frame
time_encoded_array = wts.create_time_encoded_array(
    motion_array, 
    colormap=wts.white_to_black, 
    window=20, 
    scale_factor=30, 
    offset=-30, 
    light_background=True, # sets whether the output visualization will have a light or dark background, not whether the input did
    mode='auto' # set mode to 'parallel' for longer videos
)
wts.show_video_array(time_encoded_array)

In [ ]:
# Add timestamp to video
wts.add_timestamp(time_encoded_array, black_background=False, font_scale=3, font_thickness=3, seconds_per_frame=1, in_place=True)

# Preview the created video
wts.show_video_array(time_encoded_array)

# Save the video
wts.write_mp4(time_encoded_array, "fading_trails.mp4", fps=30)

## Alternative functions

In [ ]:
# If looking at a long scan where fidelity is important, and you have a LOT of RAM available (48GB for processing chemotaxis_control.mp4), you can use fit_pixel_linear_model instead of subtract_average
residuals, _, _ = wts.fit_pixel_linear_model(video_array)
residuals[residuals > 0] = 0 # This step can often be skipped, but for worms which are darker than their background, only consider negative residuals
residuals = residuals ** 2
residuals[residuals > 255] = 255
motion_array = residuals.astype(np.uint8)
# Note: scale_factor and offset can often be set lower when using residuals instead of average subtraction (scale_factor=5, offset=0 work well for this example)

## Streaming from disk pipeline

In [ ]:
# Streaming pipeline if video is too large to load into memory
video_path = "./chemotaxis_control.mp4"

# create videocapture object
cap = cv2.VideoCapture(video_path)

# read frames one by one and create average frame
average_frame = wts.get_average_frame(cap)

In [ ]:
wts.show_time_encoding_streaming(
    video_path,
    colormap=wts.hsv_rainbow,
    window=20,
    scale_factor=10,
    offset=-10,
    light_background=True,
    window_name="preview",
    worm_length=10
)

In [ ]:
# Some example visualizations
# In this case, an early and late timepoint are shown overlaid in different colors
early = wts.get_time_encoded_frame(
    cap,
    average_frame,
    kernel_radius=21,
    scale_factor=20,
    offset=-20,
    start_frame=60*10,
    window=20,
    colormap=np.array([[255,0,0]]), # Blue trails
    light_background=True,
)
late = wts.get_time_encoded_frame(
    cap,
    average_frame,
    kernel_radius=21,
    scale_factor=20,
    offset=-20,
    start_frame=60*30,
    window=20,
    colormap=np.array([[0,0,255]]), # Red trails
    light_background=True,
)

early_and_late = np.min(np.array([early, late]), axis=0)
wts.show_frame(early_and_late)
cv2.imwrite("./early_and_late.png", early_and_late)

In [ ]:
# In this example, one timepoint is shown with fading trails
t_10_minutes = wts.get_time_encoded_frame(
    cap,
    average_frame,
    kernel_radius=21,
    scale_factor=20,
    offset=-20,
    start_frame=60*10,
    window=20,
    colormap=wts.white_to_black,
    light_background=True,
)

wts.show_frame(t_10_minutes)
cv2.imwrite("./t_10_minutes.png", t_10_minutes)

In [ ]:
# Create visualization and save it, full pipeline (note: takes 10+ minutes in this example)
wts.create_time_encoded_array_streaming(
    video_path,
    save_path="rainbow_chemotaxis.mp4",
    worm_length=10,
    scale_factor=20,
    offset=-20,
    window=20,
    colormap=wts.hsv_rainbow,
    light_background=True
)

# Hand annotations

In [ ]:
# Best for short scans (~1 minute), display motion overlayed on the original video and annotate positions and phenotypes.
# Use the arrow keys or scroll bar to select a frame. Frame 0 has no overlay and frame 1 has all motion overlayed
# Double click to add a marker at your cursor's position. Hold down any key (letter, number, ex: "r" for roaming or "1", "2", or "3" for your own phenotypes)
# To put down multiple markers for the same worm, move to a different timepoint and hold down shift while double clicking to add another marker
# Function returns a dataframe with each marker as a row, and separate columns for 
# worm ID, phenotype, x-coordinate, y-coordinate, frame number, and conversions to real units if calibration is provided

video_path = "./dying_plate.avi"
video_array = wts.read_video_file(video_path)

# set calibration for real units
cal = wts.Calibration(pixels_per_mm=985/60, frames_per_second=1)

annotations = wts.count_assist(
    video_array, 
    window_name="count assist", 
    calibration=cal
)

annotations

# Chemotaxis measurement

In [ ]:
# Quantitative measurements
# For chemotaxis, the positions, speeds, and direction of motion for each worm can be calculated at intervals over the scan
# Functions are available that do not use streaming, but streaming is almost always recommended

video_path = "./chemotaxis_control.mp4"
cal = wts.Calibration(pixels_per_mm=985/60, frames_per_second=1)

chemotaxis_data = wts.measure_chemotaxis_streaming(
    video_path,
    thresh=3, # choose based on preview results
    worm_length=10, # typical worm length in pixels
    window=10,
    interval=500,
    minimum_size=10,
    maximum_size=1000,
    test_spot=(351,564), # test spot position is also in pixel units, and the GUI has a tool to select this spot interactively
    calibration=cal,
)

#chemotaxis_data.to_csv("./chemotaxis_data.csv") # note that plate edges and markings on the plate will likely appear as artifacts and must be removed later
chemotaxis_data